In [16]:
"""
Qwen 파인튜닝 데이터셋 토큰 수 분석
- train/eval JSONL 파일의 토큰 수 분포를 확인
- Qwen3.5 토크나이저 사용 (정확한 토큰 수)
"""

import json
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer

# -------------------------
# Config
# -------------------------
MODEL_ID = "Qwen/Qwen3.5-9B"
TRAIN_PATH = "./data/7_qwen_dataset_train.jsonl"
EVAL_PATH = "./data/7_qwen_dataset_eval.jsonl"
MAX_SEQ_LENGTH = 131_072

# -------------------------
# 토크나이저 로드
# -------------------------
print("📦 토크나이저 로딩...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)


def count_tokens_for_sample(sample: dict, tokenizer) -> dict:
    """prompt(system+user) / completion(assistant) 토큰 수 분리 계산"""
    msgs = sample["messages"]
    prompt_msgs = [m for m in msgs if m["role"] != "assistant"]
    asst_msgs = [m for m in msgs if m["role"] == "assistant"]

    # prompt: system + user + generation prompt
    prompt_text = tokenizer.apply_chat_template(
        prompt_msgs,
        tokenize=False,
        add_generation_prompt=True,
    )
    completion_text = asst_msgs[0]["content"] + "<|im_end|>\n"

    p_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
    c_ids = tokenizer.encode(completion_text, add_special_tokens=False)

    return {
        "prompt_tokens": len(p_ids),
        "completion_tokens": len(c_ids),
        "total_tokens": len(p_ids) + len(c_ids),
        "channel": sample.get("metadata", {}).get("channel", ""),
        "video": sample.get("metadata", {}).get("video", ""),
        "num_instances": sample.get("metadata", {}).get("num_instances", 0),
    }


def analyze_file(path: str, label: str):
    """JSONL 파일 토큰 수 분석"""
    print(f"\n{'='*60}")
    print(f"📂 {label}: {path}")
    print(f"{'='*60}")

    samples = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            samples.append(json.loads(line))
    print(f"  총 {len(samples):,}개 샘플")

    # 토큰 수 계산
    results = []
    for s in tqdm(samples, desc="토큰 수 계산"):
        results.append(count_tokens_for_sample(s, tokenizer))

    prompt_tokens = np.array([r["prompt_tokens"] for r in results])
    completion_tokens = np.array([r["completion_tokens"] for r in results])
    total_tokens = np.array([r["total_tokens"] for r in results])

    # ---- 기본 통계 ----
    print(f"\n📊 토큰 수 통계")
    print(f"{'':>20} {'prompt':>12} {'completion':>12} {'total':>12}")
    print(f"  {'─'*56}")
    for name, arr in [("prompt", prompt_tokens), ("completion", completion_tokens), ("total", total_tokens)]:
        print(f"  {'mean':>18} {'-':>12} {'-':>12} {arr.mean():>12,.0f}" if name != "total" else "", end="")
    # 깔끔하게 다시
    print()
    for stat_name, func in [
        ("mean", np.mean), ("median", np.median), ("std", np.std),
        ("min", np.min), ("max", np.max),
        ("p5", lambda x: np.percentile(x, 5)),
        ("p25", lambda x: np.percentile(x, 25)),
        ("p75", lambda x: np.percentile(x, 75)),
        ("p90", lambda x: np.percentile(x, 90)),
        ("p95", lambda x: np.percentile(x, 95)),
        ("p99", lambda x: np.percentile(x, 99)),
    ]:
        p = func(prompt_tokens)
        c = func(completion_tokens)
        t = func(total_tokens)
        print(f"  {stat_name:>18} {p:>12,.0f} {c:>12,.0f} {t:>12,.0f}")

    # ---- MAX_SEQ_LENGTH 초과 ----
    over = (total_tokens > MAX_SEQ_LENGTH).sum()
    print(f"\n⚠️  MAX_SEQ_LENGTH({MAX_SEQ_LENGTH:,}) 초과: {over:,}개 ({over/len(results)*100:.2f}%)")

    # ---- 구간별 분포 ----
    bins = [0, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072, float("inf")]
    labels = ["~1K", "1K~2K", "2K~4K", "4K~8K", "8K~16K", "16K~32K", "32K~64K", "64K~128K", "128K+"]
    print(f"\n📈 total 토큰 구간별 분포")
    for i in range(len(bins) - 1):
        cnt = ((total_tokens >= bins[i]) & (total_tokens < bins[i + 1])).sum()
        pct = cnt / len(results) * 100
        bar = "█" * int(pct / 2)
        print(f"  {labels[i]:>10}: {cnt:>7,}개 ({pct:>5.1f}%) {bar}")

    # ---- 텔롭 없는 영상 ----
    n_empty = sum(1 for r in results if r["num_instances"] == 0)
    print(f"\n📝 텔롭 없는 영상(instances=0): {n_empty:,}개 ({n_empty/len(results)*100:.1f}%)")

    # ---- 가장 긴 샘플 top 10 ----
    sorted_idx = np.argsort(total_tokens)[::-1]
    print(f"\n🔝 가장 긴 샘플 Top 10")
    print(f"  {'#':>4} {'total':>10} {'prompt':>10} {'completion':>10}  채널/영상")
    for rank, idx in enumerate(sorted_idx[:10], 1):
        r = results[idx]
        print(f"  {rank:>4} {r['total_tokens']:>10,} {r['prompt_tokens']:>10,} {r['completion_tokens']:>10,}  {r['channel']}/{r['video'][:40]}")

    return results


# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    import os

    train_results = analyze_file(TRAIN_PATH, "Train")

    if os.path.exists(EVAL_PATH):
        eval_results = analyze_file(EVAL_PATH, "Eval")

    # ---- 전체 합산 요약 ----
    all_total = np.array([r["total_tokens"] for r in train_results])
    if os.path.exists(EVAL_PATH):
        all_total = np.concatenate([all_total, np.array([r["total_tokens"] for r in eval_results])])

    total_training_tokens = all_total.sum()
    print(f"\n{'='*60}")
    print(f"📊 전체 요약")
    print(f"  총 학습 토큰 수: {total_training_tokens:,.0f} ({total_training_tokens/1e9:.2f}B)")
    print(f"  train 총 토큰: {np.array([r['total_tokens'] for r in train_results]).sum():,.0f}")
    print(f"{'='*60}")

📦 토크나이저 로딩...

📂 Train: ./data/7_qwen_dataset_train.jsonl
  총 63,080개 샘플


토큰 수 계산: 100%|██████████| 63080/63080 [01:53<00:00, 553.82it/s] 



📊 토큰 수 통계
                           prompt   completion        total
  ────────────────────────────────────────────────────────
                mean            -            -          487                mean            -            -        1,292
                mean          487        1,292        1,779
              median          430          671        1,149
                 std          229        2,206        2,304
                 min          220            6          233
                 max        3,252      102,581      103,143
                  p5          246            6          282
                 p25          305          117          493
                 p75          614        1,661        2,254
                 p90          784        3,075        3,739
                 p95          890        4,432        5,134
                 p99        1,234        9,415       10,188

⚠️  MAX_SEQ_LENGTH(131,072) 초과: 0개 (0.00%)

📈 total 토큰 구간별 분포
         ~1K:  29,037개 ( 46.

토큰 수 계산: 100%|██████████| 3320/3320 [00:05<00:00, 556.36it/s]



📊 토큰 수 통계
                           prompt   completion        total
  ────────────────────────────────────────────────────────
                mean            -            -          491                mean            -            -        1,306
                mean          491        1,306        1,797
              median          434          682        1,152
                 std          234        2,276        2,372
                 min          228            6          241
                 max        2,186       52,150       52,807
                  p5          247            6          288
                 p25          306          119          489
                 p75          620        1,640        2,251
                 p90          788        3,114        3,845
                 p95          906        4,428        5,108
                 p99        1,303        9,783       10,422

⚠️  MAX_SEQ_LENGTH(131,072) 초과: 0개 (0.00%)

📈 total 토큰 구간별 분포
         ~1K:   1,518개 ( 45.

In [20]:
"""
16K 토큰 초과 샘플 채널별 분석
- 어떤 채널에서 잘리는지
- 채널당 몇 개 영상이 잘리는지
- 채널의 전체 영상 수 대비 비율
"""

import json
import numpy as np
from tqdm import tqdm
from collections import defaultdict
from transformers import AutoTokenizer

# -------------------------
# Config
# -------------------------
MODEL_ID = "Qwen/Qwen3.5-9B"
TRAIN_PATH = "./data/7_qwen_dataset_train.jsonl"
EVAL_PATH = "./data/7_qwen_dataset_eval.jsonl"
CUTOFF = 32768

# -------------------------
# 토크나이저
# -------------------------
print("📦 토크나이저 로딩...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)


def count_tokens(sample):
    msgs = sample["messages"]
    prompt_msgs = [m for m in msgs if m["role"] != "assistant"]
    asst_msgs = [m for m in msgs if m["role"] == "assistant"]

    prompt_text = tokenizer.apply_chat_template(
        prompt_msgs, tokenize=False, add_generation_prompt=True,
    )
    completion_text = asst_msgs[0]["content"] + "<|im_end|>\n"

    p_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
    c_ids = tokenizer.encode(completion_text, add_special_tokens=False)
    return len(p_ids), len(c_ids)


# -------------------------
# 분석
# -------------------------
# 채널별: 전체 수, 초과 수, 초과 영상 리스트
channel_total = defaultdict(int)
channel_over = defaultdict(list)  # channel -> [(video, prompt_tok, comp_tok, total_tok, num_instances)]

for path, label in [(TRAIN_PATH, "train"), (EVAL_PATH, "eval")]:
    print(f"\n📂 {label} 스캔 중...")
    with open(path, "r", encoding="utf-8") as f:
        samples = [json.loads(line) for line in f]

    for s in tqdm(samples, desc="토큰 계산"):
        ch = s["metadata"]["channel"]
        video = s["metadata"]["video"]
        num_inst = s["metadata"].get("num_instances", 0)
        channel_total[ch] += 1

        p_tok, c_tok = count_tokens(s)
        total = p_tok + c_tok

        if total > CUTOFF:
            channel_over[ch].append({
                "video": video,
                "split": label,
                "prompt_tok": p_tok,
                "completion_tok": c_tok,
                "total_tok": total,
                "num_instances": num_inst,
            })

# -------------------------
# 출력
# -------------------------
print(f"\n{'='*80}")
print(f"📊 {CUTOFF:,} 토큰 초과 샘플 채널별 분석")
print(f"{'='*80}")

total_over = sum(len(v) for v in channel_over.values())
print(f"\n총 초과 샘플: {total_over}개 / 전체 {sum(channel_total.values()):,}개")
print(f"초과 채널 수: {len(channel_over)}개 / 전체 {len(channel_total)}개\n")

# 채널별 초과 수 내림차순
sorted_channels = sorted(channel_over.items(), key=lambda x: len(x[1]), reverse=True)

print(f"{'채널':<30} {'초과':>5} {'전체':>5} {'비율':>7}  영상 목록")
print(f"{'─'*100}")

for ch, videos in sorted_channels:
    total = channel_total[ch]
    n_over = len(videos)
    pct = n_over / total * 100
    # 첫 번째 영상
    v0 = videos[0]
    print(f"{ch:<30} {n_over:>5} {total:>5} {pct:>6.1f}%  {v0['video'][:50]}")
    print(f"{'':>49}  tok={v0['total_tok']:,} (p={v0['prompt_tok']:,} c={v0['completion_tok']:,}) inst={v0['num_instances']}")
    # 나머지 영상
    for v in videos[1:]:
        print(f"{'':>49}  {v['video'][:50]}")
        print(f"{'':>49}  tok={v['total_tok']:,} (p={v['prompt_tok']:,} c={v['completion_tok']:,}) inst={v['num_instances']}")

# -------------------------
# 요약 통계
# -------------------------
all_over = [v for vlist in channel_over.values() for v in vlist]
if all_over:
    toks = [v["total_tok"] for v in all_over]
    insts = [v["num_instances"] for v in all_over]
    print(f"\n{'='*80}")
    print(f"📈 초과 샘플 통계")
    print(f"  토큰: min={min(toks):,}  max={max(toks):,}  mean={np.mean(toks):,.0f}  median={np.median(toks):,.0f}")
    print(f"  인스턴스: min={min(insts)}  max={max(insts)}  mean={np.mean(insts):.0f}  median={np.median(insts):.0f}")
    print(f"{'='*80}")

📦 토크나이저 로딩...

📂 train 스캔 중...


토큰 계산: 100%|██████████| 63080/63080 [02:03<00:00, 511.35it/s] 



📂 eval 스캔 중...


토큰 계산: 100%|██████████| 3320/3320 [00:06<00:00, 510.62it/s]


📊 32,768 토큰 초과 샘플 채널별 분석

총 초과 샘플: 26개 / 전체 66,400개
초과 채널 수: 13개 / 전체 664개

채널                                초과    전체      비율  영상 목록
────────────────────────────────────────────────────────────────────────────────────────────────────
LCK                                4   100    4.0%  '너 LPL이잖아'__g1DKmHWREkk
                                                   tok=38,482 (p=1,251 c=37,231) inst=1763
                                                   아지르 1황__fJ1EcmmJoAw
                                                   tok=45,298 (p=645 c=44,653) inst=2116
                                                   월즈의 K-아지르를 아는 사람： ㄱ―__Kbe0g4-lZNc
                                                   tok=34,122 (p=902 c=33,220) inst=1590
                                                   킅력일섬!⚡__TfJ-BN8ae1U
                                                   tok=63,873 (p=1,341 c=62,532) inst=2979
밍모                                 3   100    3.0%  게임 유튜버의 1년 현질 금액__R2NYFESoNVA
                   